<a href="https://colab.research.google.com/github/elkins/synth-dynamics/blob/main/examples/interactive_tutorials/enm_dynamics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Elastic Network Models & Langevin Dynamics

This tutorial demonstrates how to use `synth-dynamics` to predict protein flexibility using an Anisotropic Network Model (ANM) and simulate thermal fluctuations via Langevin dynamics.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q synth-dynamics biotite matplotlib
else:
    sys.path.append("../../")

import biotite.structure.io as strucio
import matplotlib.pyplot as plt
import numpy as np

## 1. Setup the System
We load a protein and define a mass-spring network between C-alpha atoms within a cutoff distance.

In [ ]:
import biotite.database.rcsb as rcsb

from synth_dynamics import ANMForceField, LangevinIntegrator, Simulation, System

# Download a small protein (e.g., Crambin, 1CRN)
pdb_file = rcsb.fetch("1CRN", "pdb", ".")
structure = strucio.load_structure(pdb_file)

# Extract only C-alpha atoms for coarse-grained dynamics
ca_atoms = structure[structure.atom_name == "CA"]
coords = ca_atoms.coord
masses = np.ones(len(coords)) * 12.0  # Coarse grained mass

# Initialize physical system
system = System(pdb_file)

# Setup Anisotropic Network Model (ANM) ForceField
# Nodes within 15 Å are connected by harmonic springs
forcefield = ANMForceField(system.equilibrium_coords, cutoff=15.0, spring_constant=1.0)

print(f"System initialized with {len(coords)} nodes.")

## 2. Run Langevin Dynamics
Langevin dynamics introduces thermal noise and friction to simulate the solvent bath.

In [ ]:
# Setup Integrator (Temperature = 300K, Friction = 1.0, Timestep = 0.01)
integrator = LangevinIntegrator(temperature=300.0, friction=1.0, dt=0.01)

# Initialize Simulation
sim = Simulation(system, forcefield, integrator)

# Run 1000 steps of dynamics
sim.run(n_steps=1000, output_path="trajectory.dcd", stride=10)

print("Simulation complete! Trajectory saved.")

## 3. Analyze Flexibility (RMSF)
Root Mean Square Fluctuation (RMSF) measures the deviation of each residue from its average position. Highly fluctuating regions are typically loops or disordered domains.

In [ ]:
import MDAnalysis
import numpy as np

# Load trajectory coordinates directly
u = MDAnalysis.coordinates.DCD.DCDReader("trajectory.dcd")
trajectory = np.array([ts.positions for ts in u])

# Calculate RMSF
mean_positions = np.mean(trajectory, axis=0)
rmsf = np.sqrt(np.mean(np.sum((trajectory - mean_positions) ** 2, axis=2), axis=0))

# Plot
plt.figure(figsize=(10, 4))
plt.plot(np.arange(1, len(rmsf) + 1), rmsf, marker="o", linestyle="-", color="teal")
plt.xlabel("Residue Index", fontsize=12)
plt.ylabel("RMSF (Å)", fontsize=12)
plt.title("Protein Flexibility Predicted by ANM Dynamics", fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()